# PEFT Recommender Experiment — Colab 一键运行

在 Colab 上跑 24 runs 的 PEFT method recommender 实验。

**预计时间**:
- L4 GPU: 约 2 小时,~10-13 计算单元
- A100: 约 1.2 小时,~17-20 计算单元
- T4: 约 3-4 小时,~7-10 计算单元(慢但便宜)

**前置要求**:
1. Colab Pro / Pro+ / Pay As You Go(免费层 T4 时长不够)
2. 在 **左侧栏 🔑 Secrets** 添加名为 `ANTHROPIC_API_KEY` 的密钥(bitext 任务的 LLM judge 需要)
3. **菜单 → Runtime → Change runtime type → 选 L4 GPU**(推荐)

**遇到问题**:断线重连后 `outputs/aggregated/all_runs.jsonl` 已经有的 run 不会重跑;参考 Cell 8 的 `--only-task` / `--only-method` 跳过已完成部分。

## Cell 1 — 验证 GPU + 配置 API key

In [ ]:
import os
import subprocess

# GPU info
print(subprocess.check_output('nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv', shell=True).decode())

# API key from Colab Secrets
from google.colab import userdata
try:
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('✓ ANTHROPIC_API_KEY loaded from Colab Secrets')
except Exception as e:
    print(f'⚠ ANTHROPIC_API_KEY not set: {e}')
    print('  → Add it via 左侧栏 🔑 Secrets to enable bitext_support runs')
    print('  → Without it, only banking77 + cuad will work (16 of 24 runs)')

## Cell 2 — 克隆 repo 并进入项目目录

首次运行用这一段。如果 Colab session 中断重连了,再跑一次也无妨(`-q` 已经存在会报错但不影响)。

In [ ]:
%cd /content
![ -d ft-recommender-experiment ] || git clone -q https://github.com/wangc22/ft-recommender-experiment.git
%cd /content/ft-recommender-experiment
!git pull -q  # 拉最新修复
!ls

## Cell 2.5 (强烈推荐) — 把 outputs/ 持久化到 Google Drive

Colab `/content` 是临时盘,runtime 重启后**所有跑过的 run 结果会丢**(模型 + 数据可以重下,但 24 个 run × 5-15 分钟的训练成果不能轻易丢)。

下面这段把 `outputs/` 软链到 Drive,之后所有 jsonl / metrics / figures 都直接写 Drive,断线 / 重启都不丢。**Cell 2 必须先跑(clone repo),再跑这一段**。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

drive_outputs = Path('/content/drive/MyDrive/ft-recommender-experiment-outputs')
drive_outputs.mkdir(parents=True, exist_ok=True)

project_outputs = Path('/content/ft-recommender-experiment/outputs')

# Migrate any existing real outputs/ contents into Drive (one-time, safe to re-run)
if project_outputs.exists() and not project_outputs.is_symlink():
    !cp -rn /content/ft-recommender-experiment/outputs/. {drive_outputs}/ 2>/dev/null || true
    !rm -rf /content/ft-recommender-experiment/outputs

# Symlink outputs → Drive (idempotent)
if not project_outputs.exists():
    project_outputs.symlink_to(drive_outputs, target_is_directory=True)

print('outputs/ → Drive symlink:', os.readlink(project_outputs))
print('Existing files in Drive:')
!ls -la {drive_outputs}/

## Cell 3 — 装依赖

Colab 自带 torch + transformers 但版本可能老,这里强制装 requirements.txt 里锁的版本。约 2-3 分钟。

In [ ]:
!pip install -q -r requirements.txt
# 验证关键依赖
!python -c "import torch, transformers, trl, peft, bitsandbytes; \
print('torch', torch.__version__, '| cuda', torch.cuda.is_available()); \
print('transformers', transformers.__version__); \
print('trl', trl.__version__); \
print('peft', peft.__version__); \
print('bitsandbytes', bitsandbytes.__version__)"

**如果 bitsandbytes import 报错**:`Runtime → Restart session`,然后从 Cell 1 重新开始跑(API key 需要重新注入,这是 Colab 行为)。

## Cell 4 — 下载 Qwen2.5-1.5B-Instruct(~1-2 分钟)

In [ ]:
!python scripts/01_download_model.py

## Cell 5 — 准备 3 个数据集(~1 分钟)

In [ ]:
!python scripts/02_prepare_all_data.py

## Cell 6 — Smoke test 在云端 GPU 上验证全链路(~3-5 分钟)

Mac 上验过 lora/dora/lora_plus,但 **qlora 必须在 CUDA 上验**。这一步顺便测 LLM judge API 通路(如果 Cell 1 加了 key)。

In [ ]:
!python scripts/03_smoke_test.py 2>&1 | tail -50

**预期看到**:
- 4 个 method × 3 个 task × `training OK` ≈ 12 行 success
- 末尾 `=== ALL SMOKE TESTS PASSED ===`
- 如果 `judge OK: means=...` 出现,LLM judge 也通了

**如果 qlora 链路报错** `bitsandbytes not compiled with CUDA`:重启 runtime 重跑 Cell 3。

## Cell 7 — 单 run 探时(banking77/lora/pilot,~1-2 分钟在 L4)

Smoke 只跑 2 步,不够估全量时间。用一个真 pilot run 看实际时长 + 验证 metrics 记录。

In [ ]:
!python -m src.training.run_one --task banking77 --method lora --scale pilot --seed 42 2>&1 | tail -40

**末尾应该看到 `=== RUN SUMMARY [banking77/pilot/lora] ===` block**,关注:
- `eval_quality_raw` 在 0.20-0.65 区间,带 ✓ 标记 → 链路正确
- `train_loss` < 4.0 → 训练健康
- `memory_cost` < 12000 MB → 显存正常

如果有 ✗ 标记或异常,**停下来 debug**,不要进 Cell 8。

## Cell 8 — 全量 24 runs(~2 小时在 L4)

正式开跑。可以让它在后台跑,Colab tab 偶尔点一下防空闲断线。

In [ ]:
!python scripts/run_matrix.py


**期间会看到**:
- 每 run 完成时一段 `=== RUN SUMMARY ===` 打印,带 sanity 标记
- 24 runs 跑完后底部一个 `=== ALL RUNS SUMMARY TABLE ===` 总表

### 断线自动续跑 ✓

`run_matrix.py` 启动时会读 `outputs/aggregated/all_runs.jsonl`,**自动跳过所有已完成的 (task, method, scale, seed) 组合**。

**断线场景**:Colab 闲置 90 分钟 / 关浏览器 / 网络抖。重连后:

1. 刷新 tab,Runtime → Reconnect
2. 跑 **Cell 1**(API key 重新注入,Colab 重启会丢)
3. 跑 **Cell 2**(回到项目目录)
4. **跳过 Cell 3-7**(依赖、模型、数据已在 disk 上,只要 runtime 没切换 GPU 类型就还在)
5. 直接再跑 **Cell 8** `!python scripts/run_matrix.py`

你会看到日志开头打印:
```
Matrix scope: 24 runs total | 12 already done (skipping) | 12 to run
```

只跑剩下的 12 个 run。

**额外开关**:
- `--force` / `--no-skip-completed`:强制重跑所有,无视已完成记录
- `--dry-run`:只打印将跳过 / 将执行的列表,不实际运行
- `--only-task X` / `--only-method Y`:只跑特定子集(配合自动 skip 一起工作)

**注意**:如果 Colab Runtime 完全 disconnect 后状态丢失(比较罕见,通常 1-2 小时不操作才发生),`models/` 和 `data/processed/` 也会丢,需要重跑 Cell 4-5。但 `all_runs.jsonl` 是 outputs/ 子目录,如果你在 Cell 11 选项 B 存了 Drive,重新挂载 Drive 可以恢复。

## Cell 9 — 分析结果 + 生成图表

In [ ]:
!python scripts/analyze_results.py

## Cell 10 — 看图表

In [ ]:
from IPython.display import Image, Markdown, display
from pathlib import Path

for f in ["rank_consistency", "cost_perf_scatter", "pilot_composite_bars",
          "memory_quality", "recommendation_summary"]:
    p = Path(f"outputs/figures/{f}.png")
    if p.exists():
        display(Markdown(f"### {f}"))
        display(Image(str(p)))
    else:
        print(f"missing: {p}")

# 也展示 markdown summary
summary_path = Path("outputs/reports/summary.md")
if summary_path.exists():
    display(Markdown("---\n## Final Recommendation Summary\n"))
    display(Markdown(summary_path.read_text()))

## Cell 11(可选)— 把结果下载到本地或 Drive

In [ ]:
# 选项 A:打包下载到本地
!tar czf experiment_results.tar.gz outputs/aggregated outputs/figures outputs/reports
from google.colab import files
files.download('experiment_results.tar.gz')

In [ ]:
# 选项 B:存到 Google Drive(更适合 24 runs 后想做长期分析)
from google.colab import drive
drive.mount('/content/drive')
!cp -r outputs /content/drive/MyDrive/ft-recommender-experiment-results-$(date +%Y%m%d)/
print('saved to Drive')

In [ ]:
# 选项 C:推回 GitHub repo(需要重新 git config + push 凭据,不推荐 Colab 第一次用)
# 这种方式适合你想把结果作为代码作品的一部分提交。
# !git config user.email 'fenglingqingning@gmail.com'
# !git config user.name 'windchime'
# !git add outputs/aggregated outputs/figures outputs/reports
# !git commit -m 'Experiment results from Colab L4'
# !git push  # 需要 PAT